# Build a Knowledge Graph for Your Site-to-Site VPN — the guided version

*A warm, step-by-step lab for network engineers. You bring the VPN. We bring the graph.*

Here is the whole truth about a VPN in one breath: **it is an overlay tunnel across an
underlay you don't own** — the public internet, or somebody else's MPLS core. Your remote
branch's service is only as reachable as the tunnel, and the tunnel is only as reachable as
the transport beneath it. Two layers, stacked, and the top one lives or dies by the bottom one.

That two-layer dependency is exactly what goes missing at 3 AM. Your tunnel monitor lights up
red and says *tunnel down* — but it never says **why**, and it never says **who hurts**. A
**knowledge graph** makes both explicit. By the end of this notebook you will have built, from
an empty page, a small graph that answers the question no `show crypto` command can:

> *"The Denver branch's circuit just dropped. Which business service is now down?"*

and watched it print **`POS Terminal`** — while also telling you the tunnel's own config was
**fine the whole time**. The thing that failed was the ISP circuit underneath it, not the crypto.

### First, calm the nerves: this is **not** machine learning
No training. No model. No embeddings. No AI guessing in the dark. A knowledge graph is just
**structured facts** (things, and the named links between them) plus **queries** that walk those
links. Everything is **deterministic and inspectable** — run it twice, get the same answer, and
you can point at every node that produced it. If you can read the output of `show crypto ipsec sa`,
you can read every line in this lab.

### What you need
Nothing. This runs in **Google Colab** with zero setup, using plain **Python + networkx +
matplotlib** (both already installed in Colab). No database, no Docker, no credentials. Press
*Runtime → Run all*, or run each cell in order with **Shift+Enter**.


## The whole vocabulary — five words, each one a VPN thing you already know

Before any code, here is the *entire* mental model. Everything after this is these five ideas,
repeated. You don't have to learn what any of them *mean* — only which VPN thing each one maps onto.

| Graph word | Plain meaning | The VPN thing it already is |
|---|---|---|
| **node** | a thing | a site, a tunnel, a transport circuit, a service |
| **edge** | a named, directed link between two nodes | "this branch is reached *via* that tunnel" |
| **label** | a node's *type* | `Site`, `Tunnel`, `Transport`, `Service` |
| **property** | a fact stored *on* a node or edge | `state='down'`, `role='spoke'`, `criticality='critical'` |
| **traversal** | walking edges to answer a question | "circuit down → which tunnel → which branch → which service?" |

That's it. Hold those five words and the rest is just your day job wearing a new hat.

One idea deserves a spotlight, because it is the whole trick: **a fact can live on an edge, not
just a node.** "The circuit is down" is not really a property of the tunnel or of the transport —
it is a property of the *relationship between them*: the tunnel **rides** a transport, and that
ride is what failed. This is the overlay/underlay lesson made literal. The tunnel's own crypto can
read `up` while the ground it stands on has fallen away. Keep an eye out for that moment — it is
where a topology diagram turns into something you can *query*.

(If you've built VXLAN over an IP fabric, you already know this shape: an overlay is only as alive
as its underlay. Same lesson, different world.)


## Setup — one import, one empty graph

**Theory (10 seconds).** `networkx` is a tiny Python library for building graphs in memory. We
will use a **`DiGraph`** — a *directed* graph, where every edge has a direction, an arrow. That
matters to us: `tun-denver → isp-denver` ("this tunnel rides that circuit") is a different
statement from `isp-denver → tun-denver`. A VPN is full of directional truth — who terminates on
whom, which overlay sits on which underlay — so directed edges are the honest choice.

Run the next cell. It imports the tools and hands you an empty graph to fill.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Both libraries ship pre-installed in Colab — nothing to pip install.
# A DiGraph is a *directed* graph: every edge is an arrow, from one node to another.
G = nx.DiGraph()

print("Empty graph ready:", G)
print("Nodes:", G.number_of_nodes(), " Edges:", G.number_of_edges())

**Checkpoint.** You should see `Empty graph ready` and `Nodes: 0  Edges: 0`. That empty `G` is
your blank whiteboard. Every step from here just adds nodes and edges to it.


## Step 1 — Sites: hub and spokes, as nodes

**Theory.** A site-to-site VPN exists to stitch places together: a central **hub** (your HQ or
data center, where the VPN concentrator and the applications live) and a set of remote **spokes**
(branches, stores, clinics). That hub-and-spoke shape is the first thing worth writing down,
because almost every interesting failure is a spoke losing its one path home.

In graph terms, each site becomes a **node** with a **label** of `Site` and a couple of
**properties**: its `role` (hub or spoke) and a `site_type`. We'll model **HQ** (the hub) plus
two branches — **Branch-Denver** and **Branch-Austin**.

Notice we are *not* dumping the device inventory in here. A node per site — not a node per switch,
AP, or interface. We store the *shape* of the design, and we'll come back to that principle by
name later.


In [ ]:
# add_node(name, **properties). role tells hub from spoke; site_type is a fact on the node.
G.add_node("HQ",            label="Site", role="hub",   site_type="datacenter")
G.add_node("Branch-Denver", label="Site", role="spoke", site_type="retail")
G.add_node("Branch-Austin", label="Site", role="spoke", site_type="retail")

print(G.number_of_nodes(), "sites so far:", list(G.nodes))
print("Facts on Branch-Denver:", G.nodes["Branch-Denver"])

**Checkpoint.** Three nodes: `HQ`, `Branch-Denver`, `Branch-Austin`, and the facts on
Branch-Denver show `role='spoke'` and `site_type='retail'`. You just wrote your VPN topology's
*actors* into a graph — no edges yet, just the places.


## Step 2 — Tunnels: the overlay, reaching each branch

**Theory.** Between each branch and HQ runs an **overlay tunnel** — an IPsec (or DMVPN/GRE-over-
IPsec) tunnel that carries the branch's traffic back to the core. That tunnel is a *thing* with a
life of its own, so it earns its own node with a `Tunnel` label. On it we store one telling
property: its **own** `state` — the crypto/config health that `show crypto ipsec sa` reports.
Right now both tunnels read `up`. Hold that thought; it's about to matter more than it looks.

(One honest caveat we're baking in: after a circuit loss, DPD/IKE keepalives *eventually* tear the
tunnel down — or, if DPD is disabled, the SA can linger indefinitely. We model the window in
between: the circuit is already down, but the crypto SA hasn't timed out yet, so the tunnel still
reports `up`. That window is exactly when a per-tunnel monitor lies to you.)

We also wire the sites to the tunnels with a **`REACHED_VIA`** edge: read `Branch-Denver
--REACHED_VIA--> tun-denver` literally as *"this branch is reachable through this tunnel."* And
because both tunnels come home to the same hub, we add a **`TERMINATES_AT`** edge from each tunnel
to HQ — the classic hub-and-spoke picture, drawn from data.

This whole step builds the **overlay layer**. The next step builds the ground it stands on — and
that is where the story turns.


In [ ]:
# The overlay: one IPsec tunnel home to HQ per branch. Its OWN state is the crypto/config health.
G.add_node("tun-denver", label="Tunnel", state="up", kind="IPsec")
G.add_node("tun-austin", label="Tunnel", state="up", kind="IPsec")

# Site --REACHED_VIA--> Tunnel: the branch is reachable *through* its tunnel.
G.add_edge("Branch-Denver", "tun-denver", rel="REACHED_VIA")
G.add_edge("Branch-Austin", "tun-austin", rel="REACHED_VIA")

# Both tunnels come home to the hub (HQ) — the classic hub-and-spoke shape.
G.add_edge("tun-denver", "HQ", rel="TERMINATES_AT")
G.add_edge("tun-austin", "HQ", rel="TERMINATES_AT")

for n, d in G.nodes(data=True):
    if d.get("label") == "Tunnel":
        print(f'{n}: state={d["state"]}, kind={d["kind"]}')

**Checkpoint.** Two tunnels print, and **both read `state=up`** — as far as the crypto is
concerned, everything is healthy. Remember that. In the next step we lay down the transport
underneath them, and you'll watch a branch's service fall over while its tunnel still insists it's up.


## Step 3 — Transports: the underlay, and the fact that lives on the edge (this is the pivotal step)

**Theory.** Here is the idea the whole lab pivots on. A tunnel does not float in space — it
**rides** a transport circuit you usually don't own: an ISP broadband hand-off, an MPLS access
port, a 4G backup. When the branch goes dark at 3 AM, the thing that failed is very often *that
circuit*, not the tunnel config. So *where* does the failure belong in our graph?

Not on the tunnel (its crypto is fine). Not on the transport node alone. It belongs on the
**relationship between them** — the *ride*. We add each circuit as a `Transport` node, then draw a
**`RIDES`** edge from tunnel to transport and hang a **`state`** property right on that edge. Read
`tun-denver --RIDES(down)--> isp-denver` literally: *"tun-denver rides isp-denver, and that ride
is down."*

The wiring, mirroring a real branch outage:

- `tun-denver → isp-denver` is **down** — the branch's broadband circuit dropped. **This one edge
  is the incident.**
- `tun-austin → mpls-austin` is **up** — Austin's MPLS access is healthy.

And here is the sharp edge of the overlay/underlay lesson: **the tunnel node `tun-denver` still
says `state='up'`.** The crypto peer, the IKE policy, the config — all fine. What died was the
ground beneath it. A monitor watching only the tunnel would tell you nothing is wrong. The graph
is about to tell you a branch just went offline.


In [ ]:
# add_edge(source, target, **properties). The circuit's reachability is a property ON the RIDES edge.
G.add_node("isp-denver",  label="Transport", kind="ISP broadband")
G.add_node("mpls-austin", label="Transport", kind="MPLS")

G.add_edge("tun-denver", "isp-denver",  rel="RIDES", state="down")  # <-- the whole incident
G.add_edge("tun-austin", "mpls-austin", rel="RIDES", state="up")

for u, v, d in G.edges(data=True):
    if d.get("rel") == "RIDES":
        print(f'{u} --RIDES({d["state"]})--> {v}   (tunnel {u} own crypto state: {G.nodes[u]["state"]})')

**Checkpoint.** Two `RIDES` edges print, and exactly one reads `down`: the
`tun-denver --RIDES(down)--> isp-denver` line. Look at the tail of that same line — the tunnel's
**own crypto state is still `up`**. That is the two-layer truth captured in one row: a healthy
overlay sitting on a dead underlay. That single edge, with a property on it, is the 3 AM event —
now a fact you can query against.


## See it — draw the graph

**Theory.** Text is great during an incident, but to *understand* a design you want the picture.
We'll colour nodes by their **label** (sites one colour, tunnels another, transports, services)
and draw the one **down** edge as a dashed red arrow so the failure jumps out. This is the same
instinct as a topology diagram — except this diagram is generated *from the data*, so it can never
drift out of sync with the truth.


In [ ]:
def draw(G, title="VPN knowledge graph"):
    colors = {"Site": "#0f7f74", "Tunnel": "#3aa0ff",
              "Transport": "#9aa5ad", "Service": "#c8702a"}
    node_colors = [colors.get(G.nodes[n].get("label"), "#cccccc") for n in G]
    pos = nx.spring_layout(G, seed=7)   # fixed seed => stable, repeatable layout

    plt.figure(figsize=(10, 7))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1900, alpha=0.92)
    nx.draw_networkx_labels(G, pos, font_size=8)

    down  = [(u, v) for u, v, d in G.edges(data=True) if d.get("state") == "down"]
    other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in down]
    # arc3 curves the arrows a touch, so reciprocal pairs never draw on top of each other.
    nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=15,
                           edge_color="#8894a0", connectionstyle="arc3,rad=0.12")
    nx.draw_networkx_edges(G, pos, edgelist=down,  arrows=True, arrowsize=15,
                           edge_color="#d34b3f", width=2.0, style="dashed",
                           connectionstyle="arc3,rad=0.12")
    nx.draw_networkx_edge_labels(G, pos, font_size=7,
        edge_labels={(u, v): d.get("rel", "") for u, v, d in G.edges(data=True)})

    plt.axis("off"); plt.title(title); plt.tight_layout(); plt.show()

draw(G)

**Checkpoint (reading the picture).** You should see the sites (teal), tunnels (blue), and
transports (grey) with `REACHED_VIA`, `TERMINATES_AT`, and `RIDES` arrows between them — and one
**dashed red** arrow from `tun-denver` to `isp-denver`, the down circuit. This is still just a
topology. It becomes a *knowledge* graph in the next step, when we add the thing a topology diagram
has never carried: the business behind the branch.


## Step 4 — The branch service: the node the tunnel monitor never had

**Theory.** This is the node your VPN gear was never designed to hold, and the reason the whole
lab exists. Your concentrator knows tunnels, SAs, peers, selectors. It has never once known that
behind `Branch-Denver` sits a **POS Terminal** — the point-of-sale system the store runs on — and
that when that branch loses its path home, **the registers stop taking payment**.

So we add a **Service** node, `POS Terminal`, with a `criticality` property, and we wire it to its
site with an **`AT_SITE`** edge: *"this service lives at that branch."* One edge — and now a
transport fact and a business fact are welded together in a place you can query. No `show` command
holds this link. It has lived nowhere except tribal knowledge. You're about to make it a
first-class, walkable fact.


In [ ]:
G.add_node("POS Terminal", label="Service", criticality="critical")
G.add_edge("POS Terminal", "Branch-Denver", rel="AT_SITE")

print("Graph now has", G.number_of_nodes(), "nodes and", G.number_of_edges(), "edges.")
print("The load-bearing link:", [f'{u} -AT_SITE-> {v}'
      for u, v, d in G.edges(data=True) if d.get("rel") == "AT_SITE"])
draw(G, title="VPN knowledge graph — now with the branch service")

**Checkpoint.** The graph has grown to **8 nodes and 7 edges**, including an orange `Service`
node, `POS Terminal`, joined by an `AT_SITE` edge to `Branch-Denver`. In the redrawn picture you
can *trace with your finger*: down circuit → tunnel → branch → service. In the next step we make
the computer trace it for you.


## Step 5 — The 3 AM question: blast radius as a traversal

**Theory.** Everything so far was setup for this. A **traversal** is just walking edges to answer
a question. Ours answers:

> *"For any tunnel whose underlay circuit is **down**, which branch services are at risk?"*

The route the walk takes:

1. find a **`RIDES`** edge whose **transport** is `state='down'` (the failed circuit);
2. hop back to the **sites** `REACHED_VIA` that tunnel;
3. from each site, hop to the **services** that sit `AT_SITE` there.

Every hop is one edge. Crucially, the query is **conditioned on the underlay state** — flip that
`RIDES` edge to `up` and step 1 finds nothing, so the whole walk returns nothing. Notice what we
*don't* check: the tunnel's own crypto state. We don't have to. The graph knows the overlay is
only as alive as the underlay it rides. Run it.


In [ ]:
def blast_radius(G):
    # Services at a branch whose tunnel RIDES a transport (circuit) that is currently down.
    at_risk = []
    for tun, transport, d in G.edges(data=True):
        # 1) a RIDES edge whose underlay transport is DOWN
        if d.get("rel") == "RIDES" and d.get("state") == "down":
            # 2) the sites reached via that tunnel
            for site, _, d2 in G.in_edges(tun, data=True):
                if d2.get("rel") != "REACHED_VIA":
                    continue
                # 3) the services sitting at that site
                for svc, _, d3 in G.in_edges(site, data=True):
                    if d3.get("rel") == "AT_SITE":
                        at_risk.append((transport, tun, site, svc))
    return at_risk

hits = blast_radius(G)
for transport, tun, site, svc in hits:
    print(f"{transport} circuit DOWN  ->  {tun} (crypto still {G.nodes[tun]['state']})  ->  {site}  ->  AT RISK: {svc}")
if not hits:
    print("nothing at risk")

**Checkpoint.** One line prints: `isp-denver circuit DOWN -> tun-denver (crypto still up) ->
Branch-Denver -> AT RISK: POS Terminal`. The monitor only ever told you a tunnel looked unhappy —
or worse, told you the tunnel was `up`
and left you hunting. Your graph just told you the **POS Terminal is at risk**, named the
**circuit** that actually failed, and reminded you the **crypto was up the whole time** — so you
don't waste the outage re-keying a tunnel that was never the problem. You got that answer because
*you* added the one node the VPN gear never had, and modelled the ride as an edge. That is the
entire pitch of a knowledge graph, and you just built it from an empty page.


## Step 6 — What-if: repair the circuit, then break it again

**Theory.** Because the failure lives on an edge as a plain property, you can *change* it and
re-ask — running "what if this circuit fails?" (or "what if the carrier fixes it?") on a
**model**, safely, with no one paged and no maintenance window. Flip the `RIDES` edge to `up`: the
blast radius clears. Flip it back to `down`: the POS Terminal returns. This is the humble beginning
of pre-incident planning — you can ask the graph what *would* break before it does.


In [ ]:
# You can read and write an edge's properties directly by [source][target].
G["tun-denver"]["isp-denver"]["state"] = "up"      # the carrier restores the circuit
print("After repair:   ", blast_radius(G) or "nothing at risk")

G["tun-denver"]["isp-denver"]["state"] = "down"    # the circuit drops again
print("After re-break: ", [svc for *_, svc in blast_radius(G)])

**Checkpoint.** After the repair you should see `nothing at risk`; after re-breaking it,
`['POS Terminal']` comes back. Same graph, same query — only the state on one edge changed. The
answer *responded*. That is what makes it a model rather than a drawing.


## Your turn #1 — a second service at the same branch

Real branches rarely run just one thing. Suppose the Denver store's **Store Wi-Fi** also depends
on that same tunnel home. Add it as a `Service` node, wire it with an `AT_SITE` edge to
`Branch-Denver`, and re-run `blast_radius`. You should now see **two** services surface from the
same down circuit — because one edge of extra truth is all it takes.

Fill in the cell below (there's a `# TODO`), then run it. The solution follows if you want to check.


In [ ]:
# TODO: add a "Store Wi-Fi" Service node (criticality="medium") that sits AT_SITE
#       "Branch-Denver", then re-run blast_radius below.

# G.add_node(...)
# G.add_edge(...)

for transport, tun, site, svc in blast_radius(G):
    print(f"AT RISK: {svc}  (at {site}, via {tun} on {transport})")

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node("Store Wi-Fi", label="Service", criticality="medium")
G.add_edge("Store Wi-Fi", "Branch-Denver", rel="AT_SITE")

for transport, tun, site, svc in blast_radius(G):
    print(f"AT RISK: {svc}  (at {site}, via {tun} on {transport})")
```

Now both `POS Terminal` **and** `Store Wi-Fi` come back from the one down circuit. The blast
radius grew the instant you told the graph one more true thing — you didn't touch the query at all.
</details>


In [ ]:
# (Solution, applied — so the rest of the notebook has both branch services in the graph.)
G.add_node("Store Wi-Fi", label="Service", criticality="medium")
G.add_edge("Store Wi-Fi", "Branch-Denver", rel="AT_SITE")
print("Services at risk now:", sorted({svc for *_, svc in blast_radius(G)}))

## Quiet reveal — you have been writing an *ontology*

Time to name the thing you've been doing. Every step, you made two very different kinds of
decision, and it's worth seeing the seam between them:

- *"A `Site` has a `role`. A `Tunnel` `RIDES` a `Transport`. A `Service` sits `AT_SITE` a `Site`."*
  — these are the **rules**: which node types exist, which edges are allowed, what shape a valid
  fact takes. That is the **schema**. Its fancy name is an **ontology** — and here's the friendly
  definition: *an ontology is the RFC of your graph, the agreed vocabulary written down as
  structure.* You already trust RFC 4301 to say what a valid IPsec security association is; an
  ontology does the same job for your graph.

- *"This particular tunnel is `tun-denver`, its crypto is `up`, and the circuit it rides is
  `down`."* — these are the **instances**: the actual data.

A rule of thumb that keeps the two straight forever: **if it has a hostname, an IP, or a circuit
ID, it is data (an instance), never schema.** `Tunnel` is schema; `tun-denver` is data. `RIDES` is
schema; "this tunnel rides isp-denver" is data. Keep the vocabulary small and stable; let the
instances be many. That single discipline is the difference between a graph you can grow for years
and a swamp you abandon in a month.


## A peek at the real thing (1/2) — the same sites and tunnels in Neo4j + Cypher

We used networkx so you could run everything with zero setup. But the *shapes* you built are
exactly what you'd build in a real graph database like **Neo4j**, whose query language is
**Cypher**. Cypher is worth a glance because it reads almost like the arrows we've been drawing —
`(node)-[:EDGE]->(node)`.

Here is **Steps 1–3** as Cypher. `MERGE` means "find this node or create it if missing" (so
re-running is safe); `SET` assigns properties *after* the match, which keeps the whole thing
idempotent. This is *reference only* — you don't run it here, it just shows the same idea in the
grown-up tool:

```cypher
MERGE (hq:Site {id: 'hq'})              SET hq.role  = 'hub';
MERGE (den:Site {id: 'branch-denver'})  SET den.role = 'spoke';

MERGE (tun:IPsecTunnel {id: 'tun-denver'})  SET tun.state = 'up';
MERGE (isp:Transport   {id: 'isp-denver'})  SET isp.kind  = 'ISP broadband';

MERGE (den)-[:REACHED_VIA]->(tun);
MERGE (tun)-[:TERMINATES_AT]->(hq);

// the incident, as a property on the relationship — same as our RIDES edge
MERGE (tun)-[r:RIDES]->(isp)
SET   r.state = 'down';
```

See it? `(tun)-[r:RIDES]->(isp)` with `SET r.state='down'` is the same statement as our
`G.add_edge("tun-denver", "isp-denver", rel="RIDES", state="down")`. Same node, same edge, same
fact-on-the-edge — one just happens to live in a database that scales to millions of nodes.

(One naming note so the Python-to-Cypher mapping is clean: our toy model used a generic `Tunnel`
node with `kind='IPsec'`, while the repo's real VPN ontology gives it a first-class `IPsecTunnel`
label — that's the richer label you see in the Cypher above. Same thing, more specific name.)


## A peek at the real thing (2/2) — the 3 AM question in Cypher

Our `blast_radius` walk is a 15-line Python function. In Cypher, that same traversal is four lines,
because in a graph database you **draw the shape you're looking for** and let the engine find every
match — no manual loops:

```cypher
MATCH (tun:IPsecTunnel)-[r:RIDES]->(c:Transport)
WHERE r.state = 'down'
MATCH (site:Site)-[:REACHED_VIA]->(tun)
MATCH (svc:Service)-[:AT_SITE]->(site)
RETURN c.id AS circuit, tun.id AS tunnel,
       site.id AS site, collect(DISTINCT svc.name) AS services_at_risk;
```

Read the first two lines as a sentence: *"match a tunnel whose RIDES-a-transport edge is down."*
That's the same step 1 you wrote in Python — the pattern you `MATCH` **is** the traversal. Run it
against a real branch dataset and `isp-denver` comes back with `POS Terminal` and `Store Wi-Fi`
among the services at risk. Different engine; identical thinking. If you can read the Python above,
you can read the Cypher — you already speak this language.


## Your turn #2 — the DMVPN twist: when the *hub* is the single point of failure

So far every failure was a spoke losing *its own* circuit. But hub-and-spoke VPNs hide a second,
sneakier failure mode. In a DMVPN, before NHRP resolves a **direct spoke-to-spoke** tunnel,
traffic from Denver to Austin **transits the hub**. So a spoke-to-spoke service depends on the
**hub** being up — *even when both spokes' own circuits are perfectly healthy.* Austin's MPLS is
green, Denver's is green, and yet Denver-to-Austin voice can still drop the moment HQ falls over.

Let's make the graph show that. We give the hub its own `hub_state`, and model a spoke-to-spoke
service with a **`TRANSITS`** edge pointing at HQ.

1. add a **`Branch VoIP`** service (`criticality="high"`) and a `TRANSITS` edge from it to `HQ`;
2. run `transits_hub_at_risk` with the hub **up** — nothing is at risk (the hub is fine);
3. set `HQ`'s `hub_state` to `"down"` and re-run — `Branch VoIP` appears, even though no spoke
   circuit changed.

Fill in the `# TODO`s, then run. The solution follows.


In [ ]:
# DMVPN hub-and-spoke: spoke-to-spoke traffic transits the HUB until a direct tunnel exists,
# so it depends on the hub even when both spokes' own circuits are healthy.
G.nodes["HQ"]["hub_state"] = "up"   # give the hub a state to toggle

# TODO 1: add Service "Branch VoIP" (criticality="high") and a TRANSITS edge
#         from "Branch VoIP" to "HQ".

# G.add_node(...)
# G.add_edge(...)

def transits_hub_at_risk(G):
    out = []
    for svc, hub, d in G.edges(data=True):
        if d.get("rel") == "TRANSITS" and G.nodes[hub].get("hub_state") == "down":
            out.append((svc, hub))
    return out

print("Hub UP  -> spoke-to-spoke at risk:", [s for s, _ in transits_hub_at_risk(G)] or "none")
# TODO 2: set G.nodes["HQ"]["hub_state"] = "down" and print transits_hub_at_risk(G) again.

<details><summary><b>Solution — click to reveal</b></summary>

```python
G.add_node("Branch VoIP", label="Service", criticality="high", pattern="spoke-to-spoke")
G.add_edge("Branch VoIP", "HQ", rel="TRANSITS")

print("Hub UP  ->", [s for s, _ in transits_hub_at_risk(G)] or "none")
G.nodes["HQ"]["hub_state"] = "down"
print("Hub DOWN->", [s for s, _ in transits_hub_at_risk(G)])
```

With the hub up, `Branch VoIP` is *in the graph* but *not at risk* — the query correctly ignores a
healthy hub. The moment HQ goes down, the exact same query surfaces it, even though neither spoke's
circuit moved. That is the DMVPN lesson a per-tunnel monitor can never give you: **the hub is a
shared dependency, and the graph makes it visible.**
</details>


In [ ]:
# (Solution, applied.)
G.add_node("Branch VoIP", label="Service", criticality="high", pattern="spoke-to-spoke")
G.add_edge("Branch VoIP", "HQ", rel="TRANSITS")

print("Hub UP   -> spoke-to-spoke at risk:", [s for s, _ in transits_hub_at_risk(G)] or "none")
G.nodes["HQ"]["hub_state"] = "down"      # the DMVPN hub goes down
print("Hub DOWN -> spoke-to-spoke at risk:", [s for s, _ in transits_hub_at_risk(G)])
G.nodes["HQ"]["hub_state"] = "up"        # restore the hub
print("Hub restored:", [s for s, _ in transits_hub_at_risk(G)] or "none")

## Make it yours — swap in *your* branch

Now the moment it becomes your tool, not mine. Change the four values below to **one** branch site,
**one** tunnel, **one** transport circuit, and **one** service *you* actually run. We add the down
`RIDES` edge for you, so your service shows up. Run it, and watch **your own service name** come
back from `blast_radius` — proof that the machine you built understands your network, not just the
demo's.

Keep it to the smallest slice that answers one real question. You can always add one more node
tomorrow — that's the whole promise of a graph you can grow.


In [ ]:
# --- change these four values to your network ---
MY_SITE      = "Clinic-Portland"
MY_TUNNEL    = "tun-portland"
MY_TRANSPORT = "isp-portland"
MY_SERVICE   = "EHR Terminal"
# ------------------------------------------------

G.add_node(MY_SITE,      label="Site", role="spoke", site_type="clinic")
G.add_node(MY_TUNNEL,    label="Tunnel", state="up", kind="IPsec")
G.add_node(MY_TRANSPORT, label="Transport", kind="ISP broadband")
G.add_node(MY_SERVICE,   label="Service", criticality="critical")

G.add_edge(MY_SERVICE, MY_SITE,      rel="AT_SITE")
G.add_edge(MY_SITE,    MY_TUNNEL,    rel="REACHED_VIA")
G.add_edge(MY_TUNNEL,  MY_TRANSPORT, rel="RIDES", state="down")   # your modelled circuit failure
G.add_edge(MY_TUNNEL,  "HQ",         rel="TERMINATES_AT")

print(f"If {MY_TRANSPORT} (the circuit under {MY_TUNNEL}) fails, these services are at risk:")
for transport, tun, site, svc in blast_radius(G):
    if site == MY_SITE:
        print(f"  AT RISK: {svc}   (at {site}, via {tun} on {transport})")

**Checkpoint.** Your own service name — `EHR Terminal`, or whatever you typed — prints as *at
risk*. That is the moment the tool stopped being a tutorial and became yours. Every other branch
you run is just four more lines away.


## The one rule that keeps this from becoming a swamp

You'll be tempted to model everything. Don't. Here is the discipline, in one line:

> **Model the nouns of your design review, not the numbers of your monitoring dashboard.**

Sites, tunnels, transports, services, hubs, failover policies — the **nouns** you'd draw on a
whiteboard while arguing about a design — those belong in the graph. Tunnel byte counters, IKE
rekey timers, per-flow latency, the raw `show crypto` dump — those are the **numbers**; leave them
in the systems that already store them well, and let the graph *reference* them when it needs to.
The graph holds the *shape of the dependency*; your NMS holds the firehose. Keep that line sharp
and your graph stays queryable for years.

### Where to go next
- **The companion Neo4j lab.** The same *idea* shows up in a real graph database — richer labels
  (`VPNService`, `IPsecTunnel`, `UnderlayTransport`, `TunnelHealth` instead of our toy
  `Site`/`Tunnel`/`Transport`/`Service`), but the same overlay-rides-underlay question — so the two
  Cypher peeks above become things you run.
- **Add the health nuance.** A tunnel can have IKE `up` but routing `down`; model both as separate
  state nodes and "crypto is up but nothing forwards" becomes one more traversal.
- **Add the change layer.** Model a `ChangeEvent` node linked to the tunnel it touched, and "what
  changed right before this branch dropped?" becomes a query, not an archaeology dig.


## You built a knowledge graph

Look back at the distance. You started with an empty page and five plain words. You added sites,
tunnels, and a down circuit — a topology. Then you added the one node the VPN gear never had, the
**branch service** — and the topology became *knowledge*. Finally you asked it the question no
`show crypto` can answer, and it printed **POS Terminal**, named the **circuit** that failed, and
reminded you the tunnel's crypto was fine the whole time — then responded correctly every time you
changed the world underneath it.

The important idea was never IPsec, and never networkx. It's this: **a VPN is an overlay on an
underlay you don't own, and operational truth has a shape** — a service sits at a branch, a branch
is reached via a tunnel, a tunnel rides a transport. Once that shape is a graph, impact analysis
stops being tribal knowledge and becomes a traversal. A runbook is a hammer. A graph you can query
is the whole toolbox.

Your tunnel monitor will always tell you *the tunnel* is down. Now you know how to build the thing
that tells you *the branch, the service, and whose circuit — not whose config — actually failed.*
Go model one real branch on your network, and ask it what breaks.
